In [ ]:
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib.cm as cm
import pandas as pd 
from gtda.time_series import SingleTakensEmbedding, takens_embedding_optimal_parameters
from sklearn.decomposition import PCA
from gtda.plotting import plot_point_cloud
from gtda.homology import VietorisRipsPersistence
from gtda.diagrams import PersistenceEntropy, Silhouette
from gtda.diagrams import BettiCurve, PersistenceLandscape
from ripser import ripser
from persim import plot_diagrams
from persim import bottleneck, bottleneck_matching, wasserstein, wasserstein_matching
from gtda.time_series import Resampler
import plotly.express as px
import math

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))

from tda_slugging.utils import read_files, find_shortest_file, batch_analyzer, mutual_information

# Data paths (type-3 slugging is committed; other types need the full 3W dataset)
DATA_3W_ALL      = str(REPO_ROOT / 'data' / '3W' / 'ALL')
DATA_3W_REAL     = str(REPO_ROOT / 'data' / '3W' / 'REAL')
DATA_3W_SIM      = str(REPO_ROOT / 'data' / '3W' / 'SIMULATED')
DATA_WELL        = str(REPO_ROOT / 'data' / 'well')
OUTPUTS_DIR      = str(REPO_ROOT / 'outputs')
IMAGES_DIR       = str(REPO_ROOT / 'images')
# Paths below require the full 3W dataset (download from https://github.com/ricardovvargas/3w_dataset):
# DATA_3W_NORMAL   = '/path/to/3w_dataset/data/0'
# DATA_3W_UNSTABLE = '/path/to/3w_dataset/data/4'


Let's look at the experimental data from Imperial College, campaign 22, where they recreate severe slugging events and measure the liquid holdup as a function of time. 

In [ ]:
C22F002 = pd.read_csv('C22F002.csv')

In [ ]:
def spectrum1(h, dt=1):
    """
    First cut at spectral estimation: very crude.
    
    Returns frequencies, power spectrum, and
    power spectral density.
    Only positive frequencies between (and not including)
    zero and the Nyquist are output.
    """
    nt = len(h)
    npositive = nt//2
    pslice = slice(1, npositive)
    freqs = np.fft.fftfreq(nt, d=dt)[pslice] 
    ft = np.fft.fft(h)[pslice]
    inv = np.fft.fft(ft)[pslice]
    psraw = np.abs(ft) ** 2
    # Double to account for the energy in the negative frequencies.
    psraw *= 2
    # Normalization for Power Spectrum
    psraw /= nt**2
    # Convert PS to Power Spectral Density
    psdraw = psraw * dt * nt  # nt * dt is record length
    return freqs, psraw, psdraw, inv

In [ ]:
fig_C22F002, data = plt.subplots(1, 3, figsize=(17, 3))

fig_C22F002.suptitle('C22F002')
data[0].set_xlabel('time (s)')
data[0].set_ylabel('liquid holdup')
data[1].set_xlabel('freq (Hz)')
data[1].set_ylabel('power spectral density')
data[1].set_xlim([0,2.5])
data[2].set_xlabel('freq')
data[2].set_ylabel('power spectral density')

freqs, ps, psd, inv = spectrum1(C22F002[C22F002.columns[1]], dt=0.1)

data[0].plot(C22F002[C22F002.columns[0]],C22F002[C22F002.columns[1]])
#data[1].loglog(freqs,psd,'r',freqs,ps,'b')
data[1].plot(freqs,psd,'b')
data[2].loglog(freqs,ps,'r')



The timeseries has clearly some (semi) periodic component every 3 to 4 seconds (<1 Hz), and other components occuring right after the main peaks with higher frequency.

In [ ]:
plt.xlim([0,20])
plt.xlabel('time (s)');
plt.ylabel('liquid holdup');
plt.plot(C22F002[C22F002.columns[0]],C22F002[C22F002.columns[1]], 'bo-')

In [ ]:
slug_signal = C22F002.iloc[:, 1]

In [ ]:
def TDAanalyze(signal, stride):
    # a function to extract persistence entropy directly, for optimal embedding parameters only
    max_time_delay = int(len(signal)/2) 
    max_embedding_dimension = int(len(signal)/20)
    print('length of signal to analyze', len(signal))
    print('max time delay', max_time_delay)
    print('max dim embedding', max_embedding_dimension)
    print('stride', stride)

    optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
        signal, max_time_delay, max_embedding_dimension, stride=stride
        )

    print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
    print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
    
    embedding_dimension = optimal_embedding_dimension
    embedding_time_delay = optimal_time_delay
    stride = stride

    embedder = SingleTakensEmbedding(
        parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
        )

    y_signal_embedded = embedder.fit_transform(signal)

    pca = PCA(n_components=3)
    y_signal_embedded_pca = pca.fit_transform(y_signal_embedded)

    plot_point_cloud(y_signal_embedded_pca)
    
    homology_dimensions = (0, 1, 2)
    VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

    y_signal_embedded_reshaped = y_signal_embedded.reshape(1, *y_signal_embedded.shape)
 
    PH_signal = VRP.fit_transform(y_signal_embedded_reshaped)
    VRP.plot(PH_signal)
    PE_signal = PersistenceEntropy()
    features = PE_signal.fit_transform(PH_signal)

    return features


def MaxPers(PH):
    # function that comptues the highest persistence (per homology class) 
    # in a presistence diagram
    arrH0 = []
    arrH2 = []
    arrH1 = []
    dist_H0 = []
    dist_H1 = []
    dist_H2 = []
    for triplet in PH:
        for i in range(len(triplet)):
            if triplet[i][2] == 0: arrH0.append(triplet[i][:2])
            if triplet[i][2] == 1: arrH1.append(triplet[i][:2])
            if triplet[i][2] == 2: arrH2.append(triplet[i][:2])
    
    for x in arrH0:
        dist = x[1]
        dist_H0.append(dist)
    for x in arrH1:
        dist = np.linalg.norm(x - [(x[0]+x[1])/2,(x[0]+x[1])/2])
        dist_H1.append(dist)
    for x in arrH2:
        dist = np.linalg.norm(x - [(x[0]+x[1])/2,(x[0]+x[1])/2])
        dist_H2.append(dist)
        
    print('Max persistence class H_0:', max(dist_H0), 'at point:', arrH0[dist_H0.index(max(dist_H0))] )
    print('Max persistence class H_1:', max(dist_H1), 'at point:', arrH1[dist_H1.index(max(dist_H1))] ) 
    print('Max persistence class H_2:', max(dist_H2), 'at point:', arrH2[dist_H2.index(max(dist_H2))] )
   
    return 

def TDAConvergenceDim(signal, stride):
    # a function that loops over the embedding dimensions 
    max_time_delay = int(len(signal)/2) 
    max_embedding_dimension = int(len(signal)/20)
    print('length of signal to analyze', len(signal))
    print('max time delay', max_time_delay)
    print('max dim embedding', max_embedding_dimension)
    print('stride', stride)

    optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
        signal, max_time_delay, max_embedding_dimension, stride=stride
        )

    print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
    print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
    
    ndim_max = optimal_embedding_dimension + 2
    features_list = []
    
    ndim = 3
    
    while ndim <= ndim_max:
    
        embedding_dimension = ndim
        embedding_time_delay = optimal_time_delay
        stride = stride

        embedder = SingleTakensEmbedding(
            parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
            )

        y_signal_embedded = embedder.fit_transform(signal)

        pca = PCA(n_components=3)
        y_signal_embedded_pca = pca.fit_transform(y_signal_embedded)

        plot_point_cloud(y_signal_embedded_pca)

        homology_dimensions = (0, 1, 2)
        VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

    # the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

        y_signal_embedded_reshaped = y_signal_embedded.reshape(1, *y_signal_embedded.shape)

        PH_signal = VRP.fit_transform(y_signal_embedded_reshaped)
        VRP.plot(PH_signal)
        PE_signal = PersistenceEntropy()
        features = PE_signal.fit_transform(PH_signal)
        
        features_to_append = ['dim', ndim, 'entropy', features]
        
        features_list.append(features_to_append)
        ndim += 1
        
    return features_list

def TDAConvergenceDelay(signal, stride):
    # a function that loops over embedding time delays
    max_time_delay = int(len(signal)/2) 
    max_embedding_dimension = int(len(signal)/20)
    print('length of signal to analyze', len(signal))
    print('max time delay', max_time_delay)
    print('max dim embedding', max_embedding_dimension)
    print('stride', stride)

    optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
        signal, max_time_delay, max_embedding_dimension, stride=stride
        )

    print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
    print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
    
    delay_max = optimal_time_delay + 4
    
    features_list = []
    
    delay = 1
    
    while delay <= delay_max:
    
        embedding_dimension = optimal_embedding_dimension
        embedding_time_delay = delay
        stride = stride

        embedder = SingleTakensEmbedding(
            parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
            )

        y_signal_embedded = embedder.fit_transform(signal)

        pca = PCA(n_components=3)
        y_signal_embedded_pca = pca.fit_transform(y_signal_embedded)

        plot_point_cloud(y_signal_embedded_pca)

        homology_dimensions = (0, 1, 2)
        VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

    # the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

        y_signal_embedded_reshaped = y_signal_embedded.reshape(1, *y_signal_embedded.shape)

        PH_signal = VRP.fit_transform(y_signal_embedded_reshaped)
        VRP.plot(PH_signal)
        PE_signal = PersistenceEntropy()
        features = PE_signal.fit_transform(PH_signal)
        
        features_to_append = ['delay', delay, 'entropy', features]
        
        features_list.append(features_to_append)
        delay += 1
        
    return features_list




In [ ]:
from sklearn.metrics import mutual_info_score

def plot_optimal_delay(X):
    max_time_delay = int(len(X)/20)
    bins = int(len(X)/100)
    mutual_information_list = []

    for time_delay in range(1, max_time_delay + 1):
        mutual_information_list.append(mutual_information(X, time_delay, n_bins=bins))
           
    plt.plot(range(1,max_time_delay+1),mutual_information_list);
    plt.xlabel('delay');
    plt.ylabel('mutual information');


Let's start the analysis of the time series by establishing the optimal parameters for the embedding.

In [ ]:
plot_optimal_delay(slug_signal)

In [ ]:
TDAConvergenceDim(slug_signal,1)

In [ ]:
TDAConvergenceDelay(slug_signal,1)

In [ ]:
print('length of signal to analyze', len(slug_signal))

max_time_delay = 15
max_embedding_dimension = 12
stride = 1

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
    slug_signal, max_time_delay, max_embedding_dimension, stride=stride
    )

print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")

In [ ]:
embedding_dimension = 8
embedding_time_delay = 11
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(slug_signal)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)


PH_slug = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(PH_slug)

In [ ]:
MaxPers(PH_slug)

In [ ]:
PE_slug = PersistenceEntropy()
features = PE_slug.fit_transform(PH_slug)
features

The embedding for the parameters suggested from the heuristics, looks very much random. There is no evident loop, circle or torus, but the max persistence and the persistent entropies for the various homology classes are quite high. Just to double check, I perform the ebmedding and the analysis again for a delay=1. 

In [ ]:
embedding_dimension = 8
embedding_time_delay = 1
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(slug_signal)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)


PH_slug = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(PH_slug)

In [ ]:
MaxPers(PH_slug)

In [ ]:
PE_slug = PersistenceEntropy()
features = PE_slug.fit_transform(PH_slug)
features

In [ ]:
SLH = Silhouette()
slug_SLH = SLH.fit_transform_plot(PH_slug)
slug_SLH

In [ ]:
Betti = BettiCurve()
Betti_wn = Betti.fit_transform_plot(PH_slug)

In [ ]:
PLand = PersistenceLandscape()
Slug_PLand = PLand.fit_transform_plot(PH_slug)
Slug_PLand

Do the same again, but with stride = 2

In [ ]:
embedding_dimension = 8
embedding_time_delay = 11
stride = 2

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(slug_signal)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)


PH_slug = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(PH_slug)

In [ ]:
PE_slug = PersistenceEntropy()
features = PE_slug.fit_transform(PH_slug)
features

In [ ]:
SLH = Silhouette()
slug_SLH = SLH.fit_transform_plot(PH_slug)
slug_SLH

In [ ]:
Betti = BettiCurve()
Betti_wn = Betti.fit_transform_plot(PH_slug)

In [ ]:
TDAanalyze(slug_signal, 1)

In [ ]:
TDAanalyze(slug_signal, 2)

Albeit there is very low persistence in this timeseries, the homology class H_2 has extremely low entropy. This is sign of order in the high dimensional structure.

In [ ]:
C22F002_downsampled_2 = C22F002.iloc[::2, :]
C22F002_downsampled_2

In [ ]:
plt.xlim([0,12])
plt.plot(C22F002_downsampled_2[C22F002_downsampled_2.columns[0]],C22F002_downsampled_2[C22F002_downsampled_2.columns[1]], 'bo-')

In [ ]:
slug_signal_down = C22F002_downsampled_2.iloc[:, 1]
slug_signal_down.size
TDAanalyze(slug_signal_down, 2)

In [ ]:
C22F002_downsampled_3 = C22F002.iloc[::3, :]
slug_signal_down = C22F002_downsampled_3.iloc[:, 1]
plt.xlim([0,12])
plt.plot(C22F002_downsampled_3[C22F002_downsampled_3.columns[0]],C22F002_downsampled_3[C22F002_downsampled_3.columns[1]], 'bo-')
TDAanalyze(slug_signal_down, 1)

In [ ]:
embedding_dimension = 5
embedding_time_delay = 8
stride = 2

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(slug_signal_down)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)


PH_slug = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(PH_slug)

In [ ]:
dgms = ripser(y_slug_embedded, maxdim=3)['dgms']
plot_diagrams(dgms, show=True)

In [ ]:
C22F046 = pd.read_csv('C22F046.csv')

fig_C22F046, data = plt.subplots(1, 3, figsize=(17, 3))

fig_C22F046.suptitle('C22F002')
data[0].set_xlabel('time (s)')
data[0].set_ylabel('liquid holdup')
data[1].set_xlabel('freq (Hz)')
data[1].set_ylabel('power spectral density')
data[1].set_xlim([0,2.5])
data[2].set_xlabel('freq')
data[2].set_ylabel('power spectral density')

freqs, ps, psd, inv = spectrum1(C22F046[C22F046.columns[1]], dt=0.1)

data[0].plot(C22F046[C22F046.columns[0]],C22F046[C22F046.columns[1]])
#data[1].loglog(freqs,psd,'r',freqs,ps,'b')
data[1].plot(freqs,psd,'b')
data[2].loglog(freqs,ps,'r')

In [ ]:
plt.xlim([0,15])
plt.plot(C22F046[C22F046.columns[0]],C22F046[C22F046.columns[1]], 'bo-')

In [ ]:
fig = px.line(title='C22F002 vs C22F046')
fig.add_scatter(x=C22F046.iloc[:, 0], y=C22F046.iloc[:, 1], name='C22F046', mode='lines+markers')
fig.add_scatter(x=C22F002.iloc[:, 0], y=C22F002.iloc[:, 1], name='C22F002', mode='lines+markers')
fig.show()

In [ ]:
C22F046_signal = C22F046.iloc[:, 1]
TDAanalyze(C22F046_signal, 1)

In [ ]:
embedding_dimension = 12
embedding_time_delay = 10
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(C22F046_signal)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
plot_optimal_delay(C22F046_signal)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)


PH_slug = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(PH_slug)

In [ ]:
PE_slug = PersistenceEntropy()
features = PE_slug.fit_transform(PH_slug)
features

In [ ]:
embedding_dimension = 10
embedding_time_delay = 1
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(C22F046_signal)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
dgms = ripser(y_slug_embedded, maxdim=3)['dgms']
plot_diagrams(dgms, show=True)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)


PH_slug = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(PH_slug)

In [ ]:
PE_slug = PersistenceEntropy()
features = PE_slug.fit_transform(PH_slug)
features

In [ ]:
embedding_dimension = 12
embedding_time_delay = 10
stride = 2

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(C22F046_signal)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
dgms = ripser(y_slug_embedded, maxdim=2)['dgms']
plot_diagrams(dgms, show=True)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)


PH_slug = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(PH_slug)

In [ ]:
PE_slug = PersistenceEntropy()
features = PE_slug.fit_transform(PH_slug)
features

In [ ]:
embedding_dimension = 10
embedding_time_delay = 3
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(C22F046_signal)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
embedding_dimension = 10
embedding_time_delay = 2
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(C22F046_signal)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)


PH_slug = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(PH_slug)

In [ ]:
PE_slug = PersistenceEntropy()
features = PE_slug.fit_transform(PH_slug)
features

In [ ]:
period = 3
periodicSampler = Resampler(period=period)

C22F046_signal = C22F046.iloc[:, 1]
C22F046_time = C22F046.iloc[:, 0]

# throw away the first 100 points
C22F046_signal = C22F046_signal[100:]
C22F046_time = C22F046_time[100:]

X_sampled, y_sampled = periodicSampler.fit_transform_resample(C22F046_time, C22F046_signal)

fig = px.line(title='Trajectory of the Lorenz solution, projected along the z-axis and resampled every 10h')
#fig.add_scatter(y=C22F046_signal, name='C22F046_signal')
fig.add_scatter(y=y_sampled, name='y_sampled', mode='lines+markers')
fig.show()

In [ ]:
TDAanalyze(y_sampled, 1)

In [ ]:
embedding_dimension = 9
embedding_time_delay = 2
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(y_sampled)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)


PH_slug = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(PH_slug)

In [ ]:
PE_slug = PersistenceEntropy()
features = PE_slug.fit_transform(PH_slug)
features

In [ ]:
TDAanalyze(y_sampled, 1)

It is pretty clear that there is not a high persistency in these data, probably because the pattern is not really periodic (not quasi-periodic), rather is very chaotic. In the equations from Perea, it is assumed that the time serie could be expanded in Fourier series. Maybe, these data are not.... 

In [ ]:
embedding_dimension = 9
embedding_time_delay = 2
stride = 1

embedder = SingleTakensEmbedding(
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
)

y_slug_embedded = embedder.fit_transform(y_sampled)

pca = PCA(n_components=3)
y_slug_embedded_pca = pca.fit_transform(y_slug_embedded)

plot_point_cloud(y_slug_embedded_pca)

In [ ]:
homology_dimensions = (0,1,2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

# the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_slug_embedded_reshaped = y_slug_embedded.reshape(1, *y_slug_embedded.shape)
print(f"y_slug_embedded_reshaped.shape",y_slug_embedded_reshaped.shape)
print(f"y_slug_embedded.shape", y_slug_embedded.shape)


PH_slug = VRP.fit_transform(y_slug_embedded_reshaped)
VRP.plot(PH_slug)

In [ ]:
MaxPers(PH_slug)

In [ ]:
def Silho():
    
    temp_list = []

    with open("list_of_csv.dat") as file:
        while (line := file.readline().rstrip()):
            temp_list.append(line)
    
    stride = 1
    PEntropy = []
    
    for filename in temp_list:
        
        print(filename)
        signal = pd.read_csv(filename)
        filename_no_extention = filename.replace('.csv','')
        
        slug_signal = []        
        slug_signal = signal.iloc[:, 1]

        optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
        slug_signal, max_time_delay, max_embedding_dimension, stride=stride
        )

        print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
        print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
       
        embedding_dimension = optimal_embedding_dimension
        embedding_time_delay = optimal_time_delay
        stride = stride

        embedder = SingleTakensEmbedding(
            parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
            )

        y_signal_embedded = embedder.fit_transform(slug_signal)#

        pca = PCA(n_components=3)
        y_signal_embedded_pca = pca.fit_transform(y_signal_embedded)

        plot_point_cloud(y_signal_embedded_pca)

        homology_dimensions = (0, 1, 2)
        VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

        # the array need to be reshaped as the library expects many 2D arrays of dimensions 
        # n_windows x window_size

        y_signal_embedded_reshaped = y_signal_embedded.reshape(1, *y_signal_embedded.shape)#

        PH_signal = VRP.fit_transform(y_signal_embedded_reshaped)
        #VRP.plot(PH_signal)
        np.savetxt(filename_no_extention + '.phg', PH_signal[0,:])
        
        PE_signal = PersistenceEntropy()
        features = PE_signal.fit_transform(PH_signal)
        PEntropy.append(features[0])
        
        SLH = Silhouette()
        slug_SLH = SLH.fit_transform(PH_signal)
        filename_silhouette = filename_no_extention + '.silh'
        np.savetxt(filename_silhouette, slug_SLH[0,:])
        np.save(filename_silhouette, slug_SLH)
        
        Betti = BettiCurve()
        slug_Betti = Betti.fit_transform(PH_signal)
        filename_betti = filename_no_extention + '.betti'
        np.savetxt(filename_betti, slug_Betti[0,:])
        
    np.savetxt('PersistentEntropy.dat',PEntropy)
        
    return

In [ ]:
Silho()

In [ ]:
plot()

In [ ]:
!cat C22F041.silh

In [ ]:
!cat PersistentEntropy.dat

In [ ]:
def Compute_PH():
    
    temp_list = []

    with open("list_of_csv_tmp.dat") as file:
        while (line := file.readline().rstrip()):
            temp_list.append(line)
    
    stride = 1
    
    for filename in temp_list:
        
        signal = pd.read_csv(filename)
        filename_no_extention = filename.replace('.csv','')
        
        slug_signal = []        
        slug_signal = signal.iloc[:, 1]

        max_time_delay = 30 #int(len(signal)/2) 
        max_embedding_dimension = 28 #int(len(signal)/22)
        optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
        slug_signal, max_time_delay, max_embedding_dimension, stride=stride
        )
  
        embedding_dimension = optimal_embedding_dimension
        embedding_time_delay = optimal_time_delay
        stride = stride

        embedder = SingleTakensEmbedding(
            parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
            )

        y_signal_embedded = embedder.fit_transform(slug_signal)
        filename_PH = filename_no_extention + '_PH'
        filename_PH_diag = filename_PH + '_diag'
        
        tmp_diag = ripser(y_signal_embedded, maxdim=2)['dgms']         
        np.save(filename_PH_diag, tmp_diag)
        print(filename_PH_diag,'dimensions', optimal_embedding_dimension, 'time delay', optimal_time_delay)

    return

In [ ]:
Compute_PH()

In [ ]:
def Compute_distances():
    
    list_of_diags = []
    save_matrix = open("bottleneck_H1.dat", "w")
    
    with open("list_of_csv.dat") as file:
        while (line := file.readline().rstrip()):
            line_name = line + '_PH_diag.npy'
            list_of_diags.append(line_name)
    
    s = (len(list_of_diags), len(list_of_diags))
    bottleneck_matrix = np.zeros(s)
    i = 0
    j = 0
    
    for first_dgrm in list_of_diags:
        i += 1
        j = 0
        for second_dgrm in list_of_diags:
            j += 1
            if j > i :
                if first_dgrm != second_dgrm:
                    diag1 = np.load(first_dgrm,allow_pickle=True)
                    diag2 = np.load(second_dgrm,allow_pickle=True)
        
                    dmg1 = list(diag1.flatten())
                    dmg2 = list(diag2.flatten())

                    d_bottle = bottleneck(dmg1[1], dmg2[1], matching=False)
                    
                    print(i-1,j-1,d_bottle)
                    save_matrix.write(str(i-1) + ' ' + str(j-1) + ' ' + str(d_bottle)+ '\n')

                    bottleneck_matrix[i-1,j-1] = d_bottle
    
    for i in range(len(list_of_diags)):
        for j in range(len(list_of_diags)):
            if bottleneck_matrix[i,j] != 0:
                bottleneck_matrix[j,i] = bottleneck_matrix[i,j]
                
                
    save_matrix.close()            
    plt.imshow(bottleneck_matrix, interpolation='nearest', cmap=cm.viridis)
    plt.colorbar()
    plt.show()

In [ ]:
Compute_distances()

Ideally, I should run a Marchenko-Pastur test to make sure this is not just noise.... 

Clerarly there is something weird happening in the timeseries nr.30, i.e. C22F038. 

In [ ]:
C22F038 = pd.read_csv('C22F038.csv')

fig_C22F038, data = plt.subplots(1, 3, figsize=(17, 3))

fig_C22F038.suptitle('C22F038')
data[0].set_xlabel('time (s)')
data[0].set_ylabel('liquid holdup')
data[1].set_xlabel('freq (Hz)')
data[1].set_ylabel('power spectral density')
data[1].set_xlim([0,2.])
data[2].set_xlabel('freq')
data[2].set_ylabel('power spectral density')

freqs, ps, psd, inv = spectrum1(C22F038[C22F038.columns[1]], dt=0.1)

data[0].plot(C22F038[C22F038.columns[0]],C22F038[C22F038.columns[1]])
data[1].plot(freqs,psd,'b')
data[2].loglog(freqs,ps,'r')

In [ ]:
C22F038_signal = C22F038.iloc[:, 1]

In [ ]:
plot_optimal_delay(C22F038_signal)

In [ ]:
max_time_delay = 30
max_embedding_dimension = 20
stride =1 

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
        signal, max_time_delay, max_embedding_dimension, stride=stride
        )

print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
        
embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = stride

embedder = SingleTakensEmbedding(    
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
    )

y_signal_embedded = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_signal_embedded_pca = pca.fit_transform(y_signal_embedded)

plot_point_cloud(y_signal_embedded_pca)

homology_dimensions = (0, 1, 2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

    # the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_signal_embedded_reshaped = y_signal_embedded.reshape(1, *y_signal_embedded.shape)

PH_signal = VRP.fit_transform(y_signal_embedded_reshaped)
VRP.plot(PH_signal)

In [ ]:
plot_point_cloud(y_signal_embedded_pca)

In [ ]:
signal = C22F038_signal

max_time_delay = 30
max_embedding_dimension = 20
stride =2 

optimal_time_delay, optimal_embedding_dimension = takens_embedding_optimal_parameters(
        signal, max_time_delay, max_embedding_dimension, stride=stride
        )

print(f"Optimal embedding time delay based on mutual information: {optimal_time_delay}")
print(f"Optimal embedding dimension based on false nearest neighbors: {optimal_embedding_dimension}")
        
embedding_dimension = optimal_embedding_dimension
embedding_time_delay = optimal_time_delay
stride = stride

embedder = SingleTakensEmbedding(    
    parameters_type="search", n_jobs=6, time_delay=embedding_time_delay, dimension=embedding_dimension, stride=stride
    )

y_signal_embedded = embedder.fit_transform(signal)

pca = PCA(n_components=3)
y_signal_embedded_pca = pca.fit_transform(y_signal_embedded)

plot_point_cloud(y_signal_embedded_pca)

homology_dimensions = (0, 1, 2)
VRP = VietorisRipsPersistence(homology_dimensions=homology_dimensions)

    # the array need to be reshaped as the library expects many 2D arrays of dimensions n_windows x window_size

y_signal_embedded_reshaped = y_signal_embedded.reshape(1, *y_signal_embedded.shape)

PH_signal = VRP.fit_transform(y_signal_embedded_reshaped)
VRP.plot(PH_signal)

Let's try to kill the frequencies >1 Hz, so to simplify the shape of the peaks. 

In [ ]:
C22F038 = pd.read_csv('C22F038.csv')

fig_C22F038, data = plt.subplots(1, 3, figsize=(17, 3))

fig_C22F038.suptitle('C22F038')
data[0].set_xlabel('time (s)')
data[0].set_ylabel('liquid holdup')
data[1].set_xlabel('freq (Hz)')
data[1].set_ylabel('power spectral density')
data[1].set_xlim([0,2.])
data[2].set_xlabel('time (s)')
data[2].set_ylabel('liquid holdup')

freqs, ps, psd, inv = spectrum1(C22F038[C22F038.columns[1]], dt=0.1)


data[0].plot(C22F038[C22F038.columns[0]],C22F038[C22F038.columns[1]])
data[1].plot(freqs,psd,'b')
data[2].plot(np.linspace(0, np.amax(C22F038[C22F038.columns[0]]), num=len(inv)), (np.real(inv)**2)/np.max(np.real(inv)**2))
#data[2].plot(C22F038[C22F038.columns[0]],C22F038[C22F038.columns[1]], 'r')


In [ ]:
# The FFT of the signal
from scipy import fftpack

sig = np.asarray(C22F038[C22F038.columns[1]])
sig_fft = fftpack.fft(sig)

# And the power (sig_fft is of complex dtype)
power = np.abs(sig_fft)**2

# The corresponding frequencies
sample_freq = fftpack.fftfreq(sig.size, d=0.1)


#sig_inv = fftpack.ifft(sig_fft)
#high_freq_fft[np.abs(sample_freq) > peak_freq] = 0
#filtered_sig = fftpack.ifft(high_freq_fft)


# Plot the FFT power
plt.figure(figsize=(6, 5))
plt.plot(sample_freq, power)
#plt.plot(sample_freq, sig_inv**2, 'r')
#plt.plot(sample_freq, sig**2, 'g')

plt.xlabel('Frequency [Hz]')
plt.ylabel('plower')
plt.xlim([0.02,2.])
plt.ylim([0,800.])

# Find the peak frequency: we can focus on only the positive frequencies
pos_mask = np.where(sample_freq > 0)
freqs = sample_freq[pos_mask]
peak_freq = freqs[power[pos_mask].argmax()]
print('peak_freq=',peak_freq)


# Check that it does indeed correspond to the frequency that we generate
# the signal with
#np.allclose(peak_freq, 1./period)

# An inner plot to show the peak frequency
#axes = plt.axes([0.55, 0.3, 0.3, 0.5])
#plt.title('Peak frequency')
#plt.plot(freqs[:8], power[:8])
#plt.setp(axes, yticks=[])

# scipy.signal.find_peaks_cwt can also be used for more advanced
# peak detection

In [ ]:
def sample_spherical(npoints, ndim=3):
    vec = np.random.randn(ndim, npoints)
    vec /= np.linalg.norm(vec, axis=0)
    return vec

In [ ]:
xi, yi, zi = sample_spherical(200)

In [ ]:
y_signal_embedded_reshaped
type(y_signal_embedded_reshaped)
y_signal_embedded_reshaped.shape

In [ ]:
sphere = []
for i in range(len(xi)):
    vec = [xi[i], yi[i], zi[i]]
    sphere.append(vec)
    
sphere = np.array(sphere)
sphere = sphere.reshape(1, *sphere.shape)

In [ ]:
sphere.shape

In [ ]:
VRP.fit_transform_plot(sphere)

In [ ]:
# fourier series defintions
tau = 0.45
def fourier16(x, a1, a2, a3, a4, a5, a6, a7, a8, a9, a10, a11, a12, a13, a14, a15, a16):
    return (a1 * np.cos(1 * np.pi / tau * x) + 
           a2 * np.cos(2 * np.pi / tau * x) + 
           a3 * np.cos(3 * np.pi / tau * x) + 
           a4 * np.cos(4 * np.pi / tau * x) + 
           a5 * np.cos(5 * np.pi / tau * x) + 
           a6 * np.cos(6 * np.pi / tau * x) + 
           a7 * np.cos(7 * np.pi / tau * x) + 
           a8 * np.cos(8 * np.pi / tau * x) +
           a9 * np.cos(9 * np.pi / tau * x) + 
           a10 * np.cos(10 * np.pi / tau * x) + 
           a11 * np.cos(11 * np.pi / tau * x) + 
           a12 * np.cos(12 * np.pi / tau * x) + 
           a13 * np.cos(13 * np.pi / tau * x) + 
           a14 * np.cos(14 * np.pi / tau * x) + 
           a15 * np.cos(15 * np.pi / tau * x) + 
           a16 * np.cos(16 * np.pi / tau * x)
           )

In [ ]:
from scipy.optimize import curve_fit

z_list = []
Ua_list = signal[75:250].tolist()

for i in range(len(Ua_list)):
    z_list.append(i/len(Ua_list))

    
z = np.asarray(z_list)
Ua = np.asarray(Ua_list)
print(type(Ua), type(z))

# plot data
fig = plt.figure()
ax1 = fig.add_subplot(111)
p1, = plt.plot(z,Ua)

# fits
popt, pcov = curve_fit(fourier16, z, Ua)

# further plots
Ua_fit8 = fourier16(z,*popt)
p2, = plt.plot(z,Ua_fit8)

plt.show()

In [ ]:
pp = signal.tolist()
pp